# 05 — Modélisation (1re partie) : baselines et régression logistique

**Données** : `train.parquet`, **mêmes 5 folds figés qu'en E6** (`make_folds`, graine 42). Familles de features : celles retenues en E6 (`features.groups`).

**Modèles comparés** :
1. `DummyClassifier(stratified)` : le plancher (hasard) ;
2. **règles E4** : score = taux de churn du segment ancienneté × usage × terminal, taux recalculé dans chaque fold ;
3. **régression logistique simple** de E6 (imputation + standardisation + one-hot) ;
4. **régression logistique améliorée** : log, winsorisation, splines, régularisation réglée.

**Métriques** (`churn.evaluation.metrics`) : ROC-AUC, PR-AUC, lift / Precision / Recall à 5, 10 et 20 %, Brier. Sur un échantillon équilibré, le lift est borné par 2 (1 / 0,5) et le Brier est celui de l'échantillon (correction du prior en E10).

In [1]:
import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display

from churn.charts import ablation_chart, coefficient_chart, publish, spline_effect_chart
from churn.config import get_config
from churn.data.split import load_train
from churn.evaluation.cv import make_folds, summarize_ablation
from churn.evaluation.metrics import roc_auc
from churn.logging_setup import setup_logging
from churn.models.baselines import SegmentRateClassifier, make_dummy
from churn.models.pipelines import load_logreg_drop_list, make_logreg_pipeline
from churn.models.preprocessing import make_improved_logreg_pipeline, spline_contribution
from churn.models.train import cross_validate_model

setup_logging()
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

cfg = get_config()
train = load_train()
y = train[cfg.data.target]
folds = make_folds(y)
drop_list = load_logreg_drop_list()
GROUPS = cfg.features.groups
print(f"{len(train)} clients, {len(folds)} folds, familles : {GROUPS}, {len(drop_list)} variables retirées (E5)")

80000 clients, 5 folds, familles : ['cycle_engagement', 'tendance_usage', 'forfait', 'compte_terminal', 'indicateurs_manquants'], 36 variables retirées (E5)


## 1. Où serait la fuite si le prétraitement était fait hors du pipeline ?

La validation croisée doit **simuler la production** : le modèle est construit sur des clients connus, puis appliqué à des clients qu'il n'a jamais vus. Tout ce qui est **appris des données** doit donc l'être sur la seule partie apprentissage de chaque fold. Si on le calcule une fois sur les 80 000 clients avant de découper :

| Étape | Ce qui est appris | Où est la fuite si c'est fait hors pipeline |
|---|---|---|
| **Imputation** | la médiane de chaque variable | la médiane inclut les clients du fold de validation : les valeurs imputées de l'apprentissage dépendent de clients « futurs » |
| **Winsorisation** | les quantiles 1 % et 99 % | les bornes d'écrêtage dépendent des extrêmes du fold de validation : un client atypique de validation déplace la borne appliquée à l'apprentissage, et lui-même est écrêté par une borne qu'il a contribué à fixer |
| **Standardisation** | moyenne et écart-type | même mécanisme : l'échelle des variables intègre la validation |
| **Encodage one-hot** | les modalités gardées (fréquence ≥ 1 %) | une modalité peut être gardée parce qu'elle est fréquente **en validation** ; le modèle a alors une colonne taillée sur des clients qu'il est censé ne pas connaître |
| **Encodage par la cible** (ici : taux de churn d'un segment) | la moyenne de `churn` par modalité | **fuite directe de la cible** : le churn du client de validation entre dans le taux de son propre segment, donc dans son propre score |

Les quatre premières fuites passent par la distribution des **variables** : avec 80 000 clients, leur effet est faible mais systématique (toujours dans le sens de l'optimisme). La dernière passe par la **cible** elle-même : c'est la plus grave. Démonstration avec le benchmark par règles :

In [2]:
leaky = SegmentRateClassifier().fit(train, y)          # taux appris sur les 80 000 clients
leaky_auc = [roc_auc(y.iloc[v], leaky.predict_proba(train.iloc[v])[:, 1]) for _, v in folds]
clean = cross_validate_model(SegmentRateClassifier(), train, y, folds, "règles (dans le fold)")
pd.DataFrame({
    "taux appris sur tout le train (fuite)": leaky_auc,
    "taux appris dans chaque fold (correct)": clean.per_fold["roc_auc"].to_numpy(),
}).assign(ecart=lambda d: d.iloc[:, 0] - d.iloc[:, 1]).describe().loc[["mean", "std", "min", "max"]].round(4)

16:58:07 | INFO    | churn.models.train | règles (dans le fold) : AUC 0.6168 ± 0.0042 (train 0.6173)


,taux appris sur tout le train (fuite),taux appris dans chaque fold (correct),ecart
mean,0.6172,0.6168,0.0004
std,0.0041,0.0042,0.0003
min,0.6107,0.6100,0.0001
max,0.6206,0.6203,0.0007


**Lecture** : calculer les taux des segments sur tout le train gonfle l'AUC de **0,4 millième** seulement (5 folds sur 5 dans le même sens). L'effet est minuscule parce que les 21 segments sont grands (au moins 274 clients) : le churn d'un client ne pèse qu'au plus 1/274 dans le taux de son propre segment. Avec un encodage par la cible sur des groupes **petits** (une modalité à 5 clients, et à l'extrême un identifiant client), le même mécanisme donnerait une AUC artificiellement proche de 1. La règle ne dépend donc pas de la taille de l'effet observé ici : **tout ce qui est appris se fait dans le fold**, et l'écart mesuré est toujours dans le sens de l'optimisme.

## 2. Baselines

In [3]:
results = {
    "Dummy (stratified)": cross_validate_model(make_dummy(), train, y, folds, "dummy"),
    "Règles E4": clean,
    "LR simple (E6)": cross_validate_model(make_logreg_pipeline(GROUPS, drop_list), train, y, folds, "LR simple"),
}
pd.DataFrame({name: r.per_fold["roc_auc"].agg(["mean", "std"]) for name, r in results.items()}).T.round(4)

16:58:08 | INFO    | churn.models.train | dummy : AUC 0.5025 ± 0.0019 (train 0.4992)


16:59:10 | INFO    | churn.models.train | LR simple : AUC 0.6542 ± 0.0038 (train 0.6593)


,mean,std
Dummy (stratified),0.5025,0.0019
Règles E4,0.6168,0.0042
LR simple (E6),0.6542,0.0038


## 3. Régression logistique améliorée : ablation des améliorations

Chaque étape ajoute une amélioration à la précédente (mêmes folds) :

1. **prétraitement de base** : imputation médiane + indicateur de manquant par colonne + standardisation, one-hot à 1 % de fréquence minimale (`handle_unknown="ignore"`) ;
2. **+ log / winsorisation** : log(1 + x) sur 30 variables à queue droite, log symétrique sur `change_mou` et `change_rev`, écrêtage aux quantiles 1 % et 99 % appris dans le fit ;
3. **+ splines** : B-splines cubiques à 5 nœuds (aux quantiles) sur `months` et `eqpdays` ;
4. **+ réglage de C** : grille L2 et L1, C ∈ {0,01 ; 0,1 ; 1 ; 10}.

**Règle de choix de C** : quand plusieurs valeurs donnent des AUC indiscernables, on retient le modèle **le plus régularisé** dont l'AUC moyenne est à moins d'un écart-type (entre folds) de la meilleure (« règle de l'écart-type »), et L1 à égalité (modèle plus parcimonieux). Choisir C sur ces mêmes folds rend l'estimation **légèrement optimiste** : la meilleure configuration profite en partie du hasard des découpages.

In [4]:
steps = {
    "LR simple (E6)": results["LR simple (E6)"],
    "prétraitement de base": cross_validate_model(make_improved_logreg_pipeline(GROUPS, drop_list, log_winsorize=False, splines=False), train, y, folds, "base"),
    "+ log / winsorisation": cross_validate_model(make_improved_logreg_pipeline(GROUPS, drop_list, splines=False), train, y, folds, "+ log/winsor"),
    "+ splines": cross_validate_model(make_improved_logreg_pipeline(GROUPS, drop_list), train, y, folds, "+ splines"),
}
grid = {}
for penalty, l1_ratio in [("L2", 0.0), ("L1", 1.0)]:
    for C in [0.01, 0.1, 1.0, 10.0]:
        grid[(penalty, C)] = cross_validate_model(
            make_improved_logreg_pipeline(GROUPS, drop_list, C=C, l1_ratio=l1_ratio), train, y, folds, f"{penalty} C={C}")
grid_table = pd.DataFrame({
    "pénalité": [p for p, _ in grid], "C": [c for _, c in grid],
    "AUC validation": [r.per_fold["roc_auc"].mean() for r in grid.values()],
    "écart-type": [r.per_fold["roc_auc"].std() for r in grid.values()],
    "AUC train": [r.per_fold["train_roc_auc"].mean() for r in grid.values()],
}).round(4)
grid_table

17:00:17 | INFO    | churn.models.train | base : AUC 0.6570 ± 0.0034 (train 0.6611)


17:00:52 | INFO    | churn.models.train | + log/winsor : AUC 0.6655 ± 0.0045 (train 0.6695)


17:01:55 | INFO    | churn.models.train | + splines : AUC 0.6691 ± 0.0038 (train 0.6733)


17:02:19 | INFO    | churn.models.train | L2 C=0.01 : AUC 0.6676 ± 0.0038 (train 0.6714)


17:02:48 | INFO    | churn.models.train | L2 C=0.1 : AUC 0.6687 ± 0.0037 (train 0.6728)


...

17:04:34 | INFO    | churn.models.train | L1 C=0.01 : AUC 0.6627 ± 0.0041 (train 0.6647)


17:05:57 | INFO    | churn.models.train | L1 C=0.1 : AUC 0.6685 ± 0.0035 (train 0.6722)


17:09:49 | INFO    | churn.models.train | L1 C=1.0 : AUC 0.6693 ± 0.0038 (train 0.6735)


17:14:19 | INFO    | churn.models.train | L1 C=10.0 : AUC 0.6694 ± 0.0038 (train 0.6737)


,pénalité,C,AUC validation,écart-type,AUC train
0,L2,0.01,0.6676,0.0038,0.6714
1,L2,0.10,0.6687,0.0037,0.6728
2,L2,1.00,0.6691,0.0038,0.6733
3,L2,10.00,0.6693,0.0038,0.6734
4,L1,0.01,0.6627,0.0041,0.6647
5,L1,0.10,0.6685,0.0035,0.6722
6,L1,1.00,0.6693,0.0038,0.6735
7,L1,10.00,0.6694,0.0038,0.6737


In [5]:
top_key = max(grid, key=lambda k: grid[k].per_fold["roc_auc"].mean())
threshold = grid[top_key].per_fold["roc_auc"].mean() - grid[top_key].per_fold["roc_auc"].std()
eligible = [k for k in grid if grid[k].per_fold["roc_auc"].mean() >= threshold]
best_key = min(eligible, key=lambda k: (k[1], 0 if k[0] == "L1" else 1))   # plus petit C, L1 à égalité
BEST_PENALTY, BEST_C = best_key
steps[f"+ réglage de C ({BEST_PENALTY}, C = {BEST_C})"] = grid[best_key]
print(f"AUC maximale : {top_key} = {grid[top_key].per_fold['roc_auc'].mean():.4f} ; seuil (max - 1 écart-type) : {threshold:.4f}")
print(f"configurations éligibles : {eligible}")
print(f"retenue (la plus régularisée) : pénalité {BEST_PENALTY}, C = {BEST_C}, AUC {grid[best_key].per_fold['roc_auc'].mean():.4f}")
no_indicator = cross_validate_model(
    make_improved_logreg_pipeline(GROUPS, drop_list, C=BEST_C, l1_ratio=1.0 if BEST_PENALTY == "L1" else 0.0,
                                  add_indicator=False), train, y, folds, "sans add_indicator")
gain_ind = grid[best_key].per_fold["roc_auc"] - no_indicator.per_fold["roc_auc"]
print(f"apport des indicateurs de manquant par colonne : {1000 * gain_ind.mean():+.1f} ± {1000 * gain_ind.std():.1f} millièmes "
      f"({int((gain_ind > 0).sum())}/5 folds en hausse)")
long = pd.concat([r.per_fold[["fold", "roc_auc"]].assign(configuration=name, modele="Régression logistique")
                  for name, r in steps.items()]).rename(columns={"roc_auc": "auc"})
ablation_lr = summarize_ablation(long, reference="LR simple (E6)")
display(ablation_lr.set_index("configuration")[["auc_moyenne", "auc_ecart_type", "gain_moyen", "gain_ecart_type", "folds_en_hausse"]].round(4))
publish(ablation_chart(ablation_lr, "LR simple (E6)",
                       "Le log et la winsorisation apportent l'essentiel du gain", order=list(steps)),
        "05_ablation_lr", width=1000)

AUC maximale : ('L1', 10.0) = 0.6694 ; seuil (max - 1 écart-type) : 0.6656
configurations éligibles : [('L2', 0.01), ('L2', 0.1), ('L2', 1.0), ('L2', 10.0), ('L1', 0.1), ('L1', 1.0), ('L1', 10.0)]
retenue (la plus régularisée) : pénalité L2, C = 0.01, AUC 0.6676


17:14:52 | INFO    | churn.models.train | sans add_indicator : AUC 0.6663 ± 0.0042 (train 0.6700)


apport des indicateurs de manquant par colonne : +1.4 ± 0.4 millièmes (5/5 folds en hausse)


,auc_moyenne,auc_ecart_type,gain_moyen,gain_ecart_type,folds_en_hausse
configuration,,,,,
+ log / winsorisation,0.6655,0.0045,0.0113,0.0023,5
"+ réglage de C (L2, C = 0.01)",0.6676,0.0038,0.0135,0.0019,5
+ splines,0.6691,0.0038,0.0149,0.0017,5
LR simple (E6),0.6542,0.0038,0.0000,0.0000,0
prétraitement de base,0.6570,0.0034,0.0028,0.0007,5


**Lecture de l'ablation** (gain fold par fold par rapport à la LR simple de E6, en millièmes d'AUC) :

| Étape | AUC | Gain cumulé | Apport de l'étape |
|---|---|---|---|
| LR simple (E6) | 0,6542 | référence | |
| prétraitement de base (one-hot 1 %, indicateurs de manquant par colonne) | 0,6570 | +2,8 | +2,8 |
| + log / winsorisation | 0,6655 | +11,3 | **+8,5** |
| + splines (`months`, `eqpdays`) | 0,6691 | +14,9 | +3,6 |
| + réglage de C (L2, C = 0,01, règle de l'écart-type) | 0,6676 | +13,5 | −1,4 |

- **Le log et la winsorisation apportent l'essentiel** (+8,5 sur +14,9) : sans eux, quelques gros consommateurs dictent les coefficients d'un modèle linéaire.
- Les **splines** ajoutent +3,6 : la forme non monotone de l'ancienneté, au-delà du seul indicateur 11-12 mois.
- Les **indicateurs de manquant par colonne** (`add_indicator`) apportent +1,4 ± 0,4, dans 5 folds sur 5 : ils signalent aussi les ratios de E6 dont le dénominateur est nul.
- **Réglage de C et L1 contre L2** : la grille est **plate**. Toutes les configurations sauf la plus régularisée en L1 (0,6627) sont entre 0,6676 et 0,6694, un écart inférieur à l'écart-type entre folds (3,8 millièmes). L1 et L2 font jeu égal à C égal (écart ≤ 0,2 millième pour C ≥ 1). Le maximum (L1, C = 10 : 0,6694) n'est pas distinguable du reste ; on retient donc, par la **règle de l'écart-type**, le modèle **le plus régularisé** dans la marge (L2, C = 0,01), au prix de 1,5 millième, pour des coefficients stables : avec C = 10, une modalité portée par un seul client recevait un coefficient de −1,9.
- **Optimisme** : C est choisi sur les mêmes folds que ceux qui servent à l'évaluer. L'effet est ici négligeable (grille plate), et le test final (E9) reste intact.

### Deux défauts du prétraitement, corrigés en cours d'étape

Une première version du prétraitement contenait deux défauts, découverts en relisant les sorties (8 coefficients exactement nuls, puis une exécution de plus de 90 minutes) :

1. **Winsorisation des indicateurs rares.** Pour un indicateur 0/1 présent chez moins de 1 % des clients (`manquant_hnd_price`, les 4 `*_was_negative`…), les quantiles 1 % et 99 % valent tous deux 0 : l'écrêtage ramenait tous les 1 à 0 et **effaçait l'information**. Même chose pour `callfwdv_Mean` et `recv_sms_Mean` (plus de 99 % de zéros). Correction : une colonne dont les deux quantiles appris sont égaux n'est pas écrêtée.
2. **Standardisation des binaires.** Une fois ces indicateurs rendus, la standardisation les transformait en valeurs énormes (un indicateur présent chez 2 clients sur 64 000 vaut environ 180 après standardisation), ce qui dégradait la convergence du solveur L1. Correction : les indicateurs 0/1 passent dans une **branche à part** (ni log, ni écrêtage, ni standardisation) et la standardisation précède l'imputation, pour que les indicateurs ajoutés par `add_indicator` restent en 0/1.

Mesure de leur effet (mêmes folds ; la cellule reproduit l'ancien comportement avec `separate_binary=False` et `keep_degenerate=False`) :

In [ ]:
variants = {
    "retenu : branche binaire séparée, écrêtage conservé": {"separate_binary": True, "keep_degenerate": True},
    "défaut 2 : binaires et indicateurs standardisés": {"separate_binary": False, "keep_degenerate": True},
    "défauts 1 et 2 : + indicateurs rares écrêtés à 0": {"separate_binary": False, "keep_degenerate": False},
}
rows = []
for name, opts in variants.items():
    l2 = cross_validate_model(make_improved_logreg_pipeline(GROUPS, drop_list, C=0.01, **opts), train, y, folds, name)
    l1 = cross_validate_model(make_improved_logreg_pipeline(GROUPS, drop_list, C=1.0, l1_ratio=1.0, **opts),
                              train, y, folds, name + " (L1)")
    fitted = make_improved_logreg_pipeline(GROUPS, drop_list, C=0.01, **opts).fit(train.iloc[folds[0][0]], y.iloc[folds[0][0]])
    Xt = fitted[:-1].transform(train.iloc[folds[0][0]])
    rows.append({
        "variante": name,
        "AUC (L2, C = 0,01)": f"{l2.per_fold['roc_auc'].mean():.4f} ± {l2.per_fold['roc_auc'].std():.4f}",
        "colonnes constantes": int((Xt.std(axis=0) == 0).sum()),
        "valeur max |x|": round(float(np.abs(Xt).max()), 1),
        "AUC (L1, C = 1)": f"{l1.per_fold['roc_auc'].mean():.4f}",
        "durée d'ajustement L1 (s / fold)": round(l1.per_fold["fit_seconds"].mean(), 1),
    })
pd.DataFrame(rows).set_index("variante")

,"AUC (L2, C = 0,01)",colonnes constantes,valeur max |x|,"AUC (L1, C = 1)",durée d'ajustement L1 (s / fold)
variante,,,,,
"retenu : branche binaire séparée, écrêtage conservé",0.6676 ± 0.0038,0,65.3,0.6693,35.0
défaut 2 : binaires et indicateurs standardisés,0.6679 ± 0.0038,0,253.0,0.6693,70.8
défauts 1 et 2 : + indicateurs rares écrêtés à 0,0.6679 ± 0.0038,8,253.0,0.6693,86.2


**Lecture** :

- **L'AUC ne change pas** (0,6676 contre 0,6679 : 0,3 millième, dans le bruit ; même 0,6693 en L1). Les colonnes effacées étaient rares, donc peu utiles au classement global.
- **Ce que les corrections changent vraiment** :
  - **8 colonnes** n'étaient plus que des constantes : leur information était perdue et leur coefficient forcé à 0 ;
  - la valeur maximale après prétraitement passe de **253 à 65**, et l'ajustement L1 de **86 s à 35 s par fold** (÷ 2,5). Le dépassement de 90 minutes de l'exécution fautive vient très probablement de la configuration la plus coûteuse (L1, C = 10) sur ces valeurs extrêmes ; seule la configuration C = 1 a été mesurée ici ;
  - le coefficient d'un indicateur se lit désormais **directement** (passage de 0 à 1), au lieu d'être exprimé par écart-type d'une variable presque toujours nulle.
- Leçon de méthode : un prétraitement peut être faux sans que l'AUC le signale. Il faut aussi contrôler les **sorties intermédiaires** (colonnes constantes, valeurs extrêmes, coefficients nuls).

## 4. Comparaison des modèles

Moyenne sur les 5 folds (± écart-type). « Écart train-validation » : AUC sur la partie apprentissage moins AUC en validation, un indicateur de surapprentissage.

In [6]:
results["LR améliorée"] = grid[best_key]
metric_names = {"roc_auc": "AUC", "pr_auc": "PR-AUC", "lift@10%": "lift@10 %", "precision@10%": "Precision@10 %",
                "recall@10%": "Recall@10 %", "brier": "Brier"}
rows = []
for name, r in results.items():
    pf = r.per_fold
    row = {"modèle": name}
    for key, label in metric_names.items():
        row[label] = f"{pf[key].mean():.4f} ± {pf[key].std():.4f}"
    row["AUC train"] = f"{pf['train_roc_auc'].mean():.4f}"
    row["écart train-validation"] = f"{(pf['train_roc_auc'] - pf['roc_auc']).mean():+.4f}"
    rows.append(row)
pd.DataFrame(rows).set_index("modèle")

,AUC,PR-AUC,lift@10 %,Precision@10 %,Recall@10 %,Brier,AUC train,écart train-validation
modèle,,,,,,,,
Dummy (stratified),0.5025 ± 0.0019,0.4969 ± 0.0010,1.0066 ± 0.0096,0.4989 ± 0.0047,0.1007 ± 0.0010,0.4974 ± 0.0019,0.4992,-0.0033
Règles E4,0.6168 ± 0.0042,0.5842 ± 0.0036,1.3064 ± 0.0151,0.6475 ± 0.0075,0.1306 ± 0.0015,0.2392 ± 0.0006,0.6173,+0.0005
LR simple (E6),0.6542 ± 0.0038,0.6256 ± 0.0047,1.4010 ± 0.0327,0.6944 ± 0.0162,0.1401 ± 0.0033,0.2321 ± 0.0009,0.6593,+0.0051
LR améliorée,0.6676 ± 0.0038,0.6482 ± 0.0043,1.4923 ± 0.0169,0.7396 ± 0.0084,0.1492 ± 0.0017,0.2286 ± 0.0009,0.6714,+0.0038


In [7]:
detail = pd.DataFrame({name: r.summary()["moyenne"] for name, r in results.items()}).T
detail[[c for c in detail.columns if "@" in c]].round(3)

,lift@5%,precision@5%,recall@5%,lift@10%,precision@10%,recall@10%,lift@20%,precision@20%,recall@20%
Dummy (stratified),0.996,0.493,0.050,1.007,0.499,0.101,1.009,0.500,0.202
Règles E4,1.340,0.664,0.067,1.306,0.648,0.131,1.249,0.619,0.250
LR simple (E6),1.445,0.716,0.072,1.401,0.694,0.140,1.350,0.669,0.270
LR améliorée,1.564,0.775,0.078,1.492,0.740,0.149,1.400,0.694,0.280


### Gain fold par fold par rapport aux règles E4

In [8]:
rules_auc = results["Règles E4"].per_fold["roc_auc"].to_numpy()
gains = pd.DataFrame({name: r.per_fold["roc_auc"].to_numpy() - rules_auc
                      for name, r in results.items() if name != "Règles E4"})
gains.index.name = "fold"
display((1000 * gains).round(1))
(1000 * gains).agg(["mean", "std", "min"]).round(1).rename(index={"mean": "gain moyen (millièmes)", "std": "écart-type", "min": "pire fold"})

,Dummy (stratified),LR simple (E6),LR améliorée
fold,,,
0,-111.8,38.0,51.6
1,-117.5,32.2,48.2
2,-108.5,40.6,51.9
3,-118.0,35.6,50.1
4,-115.8,40.5,52.4


,Dummy (stratified),LR simple (E6),LR améliorée
gain moyen (millièmes),-114.3,37.4,50.8
écart-type,4.1,3.6,1.7
pire fold,-118.0,32.2,48.2


**Lecture** :

- **Dummy** : AUC 0,503, lift 1,0 ; c'est le hasard, et il est battu de 114 millièmes par les règles.
- **Règles E4** : AUC 0,617, lift@10 % 1,31 : trois critères simples captent déjà une bonne part du signal.
- **LR améliorée** : AUC **0,668 ± 0,004**, PR-AUC 0,648, **lift@10 % 1,49** (Precision@10 % 74 %, contre 65 % pour les règles) ; elle bat les règles de **+50,8 ± 1,7 millièmes, dans les 5 folds** (pire fold : +48,2). À capacité égale (10 % des clients contactés), elle atteint 14,9 % des churners contre 13,1 %.
- **Surapprentissage** : écart train-validation de +3,8 millièmes pour la LR améliorée, contre +5,1 pour la LR simple : la régularisation (C = 0,01) tient.
- **Brier** (0,229) : calculé sur l'échantillon équilibré, il ne dit rien encore de la calibration réelle (E10).
- Le lift@10 % est borné par 2 sur cet échantillon (taux de base 50 %) : 1,49 représente près de la moitié du maximum possible.

## 5. Coefficients de la meilleure régression logistique

Modèle réajusté sur les 80 000 clients du train. Les variables numériques étant standardisées, un coefficient se lit en **log-odds par écart-type** ; pour une modalité one-hot, par rapport à l'absence de cette modalité. Les bases de splines sont exclues du classement (un coefficient isolé n'a pas de sens) et présentées sous forme de courbe.

In [9]:
best_model = make_improved_logreg_pipeline(GROUPS, drop_list, C=BEST_C,
                                           l1_ratio=1.0 if BEST_PENALTY == "L1" else 0.0).fit(train, y)
names = best_model.named_steps["preprocess"].get_feature_names_out()
coefs = pd.Series(best_model.named_steps["model"].coef_.ravel(), index=names)
readable = coefs.rename(lambda n: n.split("__", 1)[1])
infrequent = readable[readable.index.str.contains("infrequent")]
eligible_coefs = readable[~coefs.index.str.startswith("spline__") & ~readable.index.str.contains("infrequent")]
top15 = eligible_coefs.reindex(eligible_coefs.abs().sort_values(ascending=False).index).head(15)
print(f"{len(coefs)} coefficients, dont {int((coefs == 0).sum())} exactement nuls")
print("modalités regroupées « infréquentes » (< 1 % des clients), exclues du classement :")
display(infrequent.round(3).to_frame("coefficient"))
display(top15.round(3).to_frame("coefficient"))
publish(coefficient_chart(top15, "Terminal ancien et fin d'engagement poussent au churn ; un usage élevé le freine"),
        "05_coefficients", width=1000)

202 coefficients, dont 0 exactement nuls
modalités regroupées « infréquentes » (< 1 % des clients), exclues du classement :


,coefficient
crclscod_infrequent_sklearn,0.002
area_infrequent_sklearn,0.025
dualband_infrequent_sklearn,-0.025
refurb_new_infrequent_sklearn,-0.004
hnd_webcap_infrequent_sklearn,-0.010
infobase_infrequent_sklearn,-0.009
HHstatin_infrequent_sklearn,0.048
dwllsize_infrequent_sklearn,-0.027


,coefficient
handset_old,0.565
in_contract_end,0.536
mou_Mean,-0.373
missingindicator_change_mou,0.329
missingindicator_change_rev,0.329
avgqty,0.321
crclscod_EA,-0.266
area_NORTHWEST/ROCKY MOUNTAIN AREA,0.218
totmrc_Mean,-0.156
area_DC/MARYLAND/VIRGINIA AREA,-0.154


**Top 15 des coefficients et comparaison avec l'EDA** (continues : par écart-type ; indicateurs et modalités : passage de 0 à 1) :

| Variable | Coef. | Lecture | Cohérent avec l'EDA ? |
|---|---|---|---|
| `handset_old` | +0,565 | terminal de 300 jours ou plus | oui : 56,3 % contre 39,4 % (E6) |
| `in_contract_end` | +0,536 | ancienneté de 11-12 mois | oui : 63,5 % contre 47,3 % (E4, E6) |
| `mou_Mean` | −0,373 | plus d'usage récent, moins de churn | oui : 58,9 % au 1er décile, 44,4 % au dernier (E4) |
| `missingindicator_change_mou` / `_change_rev` | +0,329 chacun | variation d'usage absente | oui (76 % de churn), mais **signal « déjà parti » suspect (D32)** ; le coefficient se partage entre deux colonnes identiques |
| `avgqty` | +0,321 | appels moyens depuis l'ouverture, **à usage récent égal** | effet conditionnel : un client qui appelait plus avant qu'aujourd'hui est en **baisse d'usage** ; cohérent avec E4 (baisse d'usage, +10 pts) |
| `crclscod_EA` | −0,266 | classe de crédit EA | oui : classe E à 39,7 % (E4) |
| `area_NORTHWEST/ROCKY MOUNTAIN` | +0,218 | région | oui : région au churn le plus élevé, 56,1 % (E4) |
| `totmrc_Mean` | −0,156 | forfait mensuel plus élevé, moins de churn | oui : 43-44 % au-delà de 60 $ contre 53-57 % sous 30 $ (E4) |
| `area_DC/MARYLAND/VIRGINIA` | −0,154 | région | oui : région au churn le plus bas, 45,9 % (E4) |
| `asl_flag_Y` | −0,150 | limite de dépense sur le compte | oui : 41,1 % contre 50,9 % (E4) |
| `missingindicator_hnd_price` / `manquant_hnd_price` | −0,140 chacun | prix du terminal inconnu | oui : 37,6 % contre 49,7 % (E6) ; indicateur en double, coefficient partagé |
| `area_SOUTH FLORIDA` | +0,133 | région | oui : 53,8 % (E4) |
| `drop_vce_Mean` | +0,132 | appels coupés, **à usage égal** | effet invisible en univarié (les gros utilisateurs ont plus d'appels coupés **et** churnent moins) ; une fois l'usage pris en compte, la mauvaise qualité réseau pousse au churn |

Aucun signe n'est en contradiction avec l'EDA. Deux effets n'apparaissent qu'**à autres variables égales** (`avgqty`, `drop_vce_Mean`) : c'est l'intérêt d'un modèle multivarié. Les coefficients restent des **associations** conditionnelles, pas des causes.

In [10]:
effects = {
    "months": spline_contribution(best_model, "months", np.arange(6, 61)),
    "eqpdays": spline_contribution(best_model, "eqpdays", np.arange(0, 1501, 10)),
}
publish(spline_effect_chart(effects, "Les splines retrouvent les formes vues en EDA",
                            {"months": "ancienneté (mois)", "eqpdays": "âge du terminal (jours)"}),
        "05_splines", width=1100)
pd.DataFrame({"months": effects["months"].loc[[6, 9, 11, 12, 15, 24, 36, 60]].round(2)})

,months
6,-0.40
9,-0.07
11,0.18
12,0.25
15,0.28
24,0.11
36,-0.03
60,-0.13


**Lecture des splines** (contribution au log-odds, à autres variables égales, **en plus** des indicateurs `in_contract_end` et `handset_old`) :

- **`months`** : contribution basse à 6 mois (−0,40), qui monte jusqu'à un maximum vers **14 mois** (+0,29), puis décroît régulièrement (−0,13 à 60 mois). Ajoutée à `in_contract_end` (+0,54 à 11-12 mois), elle reproduit la courbe de l'EDA : churn faible sous engagement, pic à l'échéance, décroissance avec la fidélité.
- **`eqpdays`** : la spline plonge juste après 300 jours (−0,17 vers 400 jours) puis remonte (+0,13 au-delà de 1 200 jours). Elle **complète** `handset_old` (+0,57 dès 300 jours) : la marche est portée par l'indicateur, la spline ajuste la forme de part et d'autre. L'effet total de l'âge du terminal est la somme des deux, et reste croissant après 300 jours.

## Conclusion

1. **Modèle de référence pour E8** : régression logistique améliorée (L2, C = 0,01), **AUC 0,668 ± 0,004**, lift@10 % 1,49 ; **+50,8 millièmes sur les règles, dans les 5 folds**.
2. Le gain sur la LR simple (+13,5) vient surtout du **log et de la winsorisation** (+8,5), puis des **splines** (+3,6) ; le réglage de C n'apporte rien (grille plate).
3. Deux défauts du prétraitement ont été corrigés (voir la section dédiée) : ils ne changeaient pas l'AUC, mais effaçaient des informations et rendaient le modèle lent et ses coefficients faux.
4. Les coefficients sont **tous cohérents avec l'EDA** ; deux effets n'apparaissent qu'à autres variables égales (baisse d'usage via `avgqty`, qualité réseau via `drop_vce_Mean`).
5. **Point pour E8 (D32)** : `add_indicator` réintroduit le signal « déjà parti » (`missingindicator_change_mou`, +0,33) alors que la famille `deja_parti` est exclue. Le test de fuite devra aussi neutraliser ces indicateurs.
6. Écart restant avec LightGBM par défaut (0,690 en E6) : 22 millièmes. C'est l'objet de E8 : ce que les arbres gagnent en interactions, et ce qu'on perd en lisibilité.

# 2e partie — Modèles avancés et test de fuite (E8)

Cette partie s'exécute de façon autonome (cellule de préparation ci-dessous) : mêmes données, **mêmes 5 folds figés** que E6 et la 1re partie. Tous les modèles sont construits par `build_pipeline(model_name)` et évalués par `cross_validate_model`.

In [ ]:
import time

import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display

from churn.charts import model_comparison_chart, publish
from churn.config import get_config
from churn.data.split import load_train
from churn.evaluation.cv import make_folds
from churn.logging_setup import setup_logging
from churn.models.factory import build_pipeline
from churn.models.leakage import active_client_mask, subset_auc_per_fold
from churn.models.train import cross_validate_model

setup_logging()
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.width", 230)
pd.set_option("display.max_columns", 30)

cfg = get_config()
train = load_train()
y = train[cfg.data.target]
folds = make_folds(y)
active = active_client_mask(train)
print(f"{len(train)} clients, {len(folds)} folds ; clients actifs : {int(active.sum())} ({active.mean():.1%})")

80000 clients, 5 folds ; clients actifs : 79282 (99.1%)


## 6. Comparaison des modèles

| Modèle | Pourquoi on l'utilise | Avantages sur ce problème | Limites sur ce problème |
|---|---|---|---|
| **Arbre de décision** (profondeur 5, feuilles ≥ 100 clients) | Référence pédagogique : un modèle non linéaire qu'on peut dessiner | Règles « si… alors » lisibles ; capte seul les seuils (11-12 mois, 300 jours) et les NaN | Instable (un petit changement de données change l'arbre) ; au plus 32 groupes de clients, donc un score en marches ; faible sur un signal diffus |
| **Forêt aléatoire** (300 arbres, feuilles ≥ 50) | Moyenne de nombreux arbres décorrélés (*bagging*) : réduit la variance d'un arbre seul | Robuste, peu de réglages, capte les interactions | Probabilités peu calibrées ; plus lente et plus lourde ; en général moins précise que le boosting sur données tabulaires |
| **LightGBM** (défaut) | Boosting de gradient : chaque arbre corrige les erreurs des précédents | Rapide ; NaN et catégorielles natifs ; très performant sur données tabulaires ; 0,690 dès E6 | Peut surapprendre s'il est mal réglé ; boîte noire (explication par SHAP en E11) |
| **CatBoost** (500 itérations, catégorielles natives) | Boosting à arbres symétriques, avec un encodage des catégorielles conçu pour éviter la fuite de la cible (statistiques ordonnées) | Bon réglage par défaut ; gère `crclscod`, `area`… sans one-hot | Plus lent ; nombreuses options ; boîte noire |

Les baselines (Dummy, règles) et la LR améliorée (L2, C = 0,01) de la 1re partie sont réévaluées ici dans le même tableau.

In [ ]:
MODELS = {
    "Dummy": ("dummy", {}),
    "Règles E4": ("rules", {}),
    "LR améliorée": ("logreg", {}),
    "Arbre de décision": ("decision_tree", {}),
    "Forêt aléatoire": ("random_forest", {}),
    "LightGBM": ("lightgbm", {}),
    "CatBoost": ("catboost", {}),
}
FOLD_LIMIT_SECONDS = 300
results = {}
for label, (name, params) in MODELS.items():
    results[label] = cross_validate_model(build_pipeline(name, **params), train, y, folds, label)
    worst = results[label].per_fold["fit_seconds"].max()
    if worst > FOLD_LIMIT_SECONDS:
        print(f"{label} : un fold a pris {worst:.0f} s (> 5 min) ; relance avec moins d'itérations")
        results[label] = cross_validate_model(build_pipeline(name, iterations=250), train, y, folds, label)
print("durée d'entraînement maximale par fold :",
      {k: round(r.per_fold['fit_seconds'].max(), 1) for k, r in results.items()})

durée d'entraînement maximale par fold : {'Dummy': np.float64(0.0), 'Règles E4': np.float64(12.5), 'LR améliorée': np.float64(2.8), 'Arbre de décision': np.float64(2.6), 'Forêt aléatoire': np.float64(9.3), 'LightGBM': np.float64(1.1), 'CatBoost': np.float64(50.3)}


In [ ]:
lr_auc = results["LR améliorée"].per_fold["roc_auc"].to_numpy()
rows = []
for label, r in results.items():
    pf = r.per_fold
    gain = 1000 * (pf["roc_auc"].to_numpy() - lr_auc)
    rows.append({
        "modèle": label,
        "auc": pf["roc_auc"].mean(), "auc_std": pf["roc_auc"].std(),
        "PR-AUC": pf["pr_auc"].mean(), "lift@10 %": pf["lift@10%"].mean(),
        "Precision@10 %": pf["precision@10%"].mean(), "Brier": pf["brier"].mean(),
        "AUC train": pf["train_roc_auc"].mean(),
        "écart train-validation": (pf["train_roc_auc"] - pf["roc_auc"]).mean(),
        "entraînement (s / fold)": pf["fit_seconds"].mean(),
        "gain vs LR (millièmes)": gain.mean(), "gain σ": gain.std(ddof=1),
        "folds gagnés sur 5": int((gain > 0).sum()),
    })
comparison = pd.DataFrame(rows).set_index("modèle")
display(comparison.round({"auc": 4, "auc_std": 4, "PR-AUC": 4, "lift@10 %": 3, "Precision@10 %": 3, "Brier": 4,
                          "AUC train": 4, "écart train-validation": 4, "entraînement (s / fold)": 1,
                          "gain vs LR (millièmes)": 1, "gain σ": 1}))
publish(model_comparison_chart(comparison, "Le boosting devance la forêt, la LR améliorée et l'arbre de décision",
                               reference="LR améliorée"), "05_comparaison_modeles", width=1000)

,auc,auc_std,PR-AUC,lift@10 %,Precision@10 %,Brier,AUC train,écart train-validation,entraînement (s / fold),gain vs LR (millièmes),gain σ,folds gagnés sur 5
modèle,,,,,,,,,,,,
Dummy,0.5025,0.0019,0.4969,1.007,0.499,0.4974,0.4992,-0.0033,0.0,-165.1,3.3,0
Règles E4,0.6168,0.0042,0.5842,1.306,0.648,0.2392,0.6173,0.0005,8.1,-50.8,1.7,0
LR améliorée,0.6676,0.0038,0.6482,1.492,0.740,0.2286,0.6714,0.0038,2.7,0.0,0.0,0
Arbre de décision,0.6391,0.0023,0.6119,1.436,0.712,0.2341,0.6468,0.0076,2.5,-28.5,4.6,0
Forêt aléatoire,0.6781,0.0055,0.6599,1.515,0.751,0.2266,0.7855,0.1074,9.1,10.4,3.2,5
LightGBM,0.6917,0.0043,0.6766,1.572,0.779,0.2216,0.7689,0.0772,1.0,24.1,3.5,5
CatBoost,0.6929,0.0038,0.6773,1.576,0.781,0.2213,0.7936,0.1007,48.9,25.2,3.1,5


**Lecture** (5 folds figés, gain fold par fold par rapport à la LR améliorée) :

- **Boosting en tête** : CatBoost **0,6929 ± 0,0038** et LightGBM **0,6917 ± 0,0043**, chacun devant la LR dans les **5 folds** (+25,2 et +24,1 millièmes). Entre eux, 1,2 millième d'écart, **moins que l'écart-type entre folds** : ils sont indiscernables.
- **Forêt aléatoire** : 0,6781, +10,4 sur la LR (5/5), mais 14 millièmes derrière le boosting : la moyenne d'arbres indépendants capte moins bien un signal diffus que des arbres qui corrigent les erreurs des précédents.
- **Arbre de décision (profondeur 5)** : 0,6391, sous la LR dans les 5 folds (−28,5) ; il bat quand même les règles (+22 millièmes), avec des règles tout aussi lisibles.
- **Surapprentissage** : écart train-validation de +0,077 (LightGBM), +0,101 (CatBoost) et +0,107 (forêt), contre +0,004 pour la LR. Les modèles d'arbres mémorisent une partie du train avec leurs réglages par défaut ; leur AUC de validation reste pourtant la meilleure. Le réglage des hyperparamètres (E9) devra réduire cet écart.
- **Calibration** (Brier sur l'échantillon équilibré) : 0,221 pour le boosting, 0,229 pour la LR ; écarts faibles, la calibration réelle est traitée en E10.
- **Temps** : LightGBM 1,0 s par fold, CatBoost 48,9 s (sous la limite de 5 minutes, itérations non réduites).
- **Lift@10 %** : 1,58 pour le boosting contre 1,49 pour la LR : parmi les 10 % de clients les mieux classés, 78 % de churners contre 74 %.

## 7. Test de fuite « client déjà parti » (D32, D57, D66)

**Hypothèse** : un client sans aucune donnée d'usage, ou sans variation d'usage calculable, est peut-être **déjà parti** au moment de la mesure. Le modèle apprendrait alors une **conséquence** du churn. En production, ce signal ne servirait à rien : un client déjà inactif n'a pas besoin d'un modèle pour être repéré.

Deux contrôles, sur les **2 meilleurs modèles** du tableau précédent :

- **(a) version neutralisée** : `LeakNeutralizer`, en tête de pipeline, impute le bloc d'usage (`rev_Mean`, `mou_Mean`, …) et `change_mou` / `change_rev` par leur **médiane apprise dans le fold**. Plus aucune absence n'est observable : ni indicateur `manquant_*` ou `missingindicator_*`, ni NaN exploitable par un arbre, ni ratio de E6 trahissant l'absence (`overage_share_rev`…). La famille `deja_parti` est déjà exclue de la configuration (D57) ;
- **(b) clients actifs** : AUC recalculée sur les seuls clients de validation avec un usage mesuré et un `change_*` renseigné (99,1 % des clients).

**Règle de décision (fixée avant de voir les résultats)** : si la perte de la version neutralisée (sur tous les clients) est **inférieure à l'écart-type de l'AUC entre folds**, on garde la version complète et on documente ; sinon, l'AUC « clients actifs » devient la métrique métier de référence et on discute la version à déployer.

In [ ]:
candidates = comparison.drop(index=["Dummy", "Règles E4"]).sort_values("auc", ascending=False).index[:2].tolist()
print("2 meilleurs modèles :", candidates)
leak_rows, neutral_results = [], {}
for label in candidates:
    name, params = MODELS[label]
    neutral_results[label] = cross_validate_model(build_pipeline(name, neutralize_leak=True, **params),
                                                  train, y, folds, f"{label} neutralisé")
    for version, r in [("complet", results[label]), ("neutralisé", neutral_results[label])]:
        all_auc = r.per_fold["roc_auc"].to_numpy()
        act_auc = subset_auc_per_fold(r.oof, y, folds, active)
        leak_rows.append({"modèle": label, "version": version,
                          "AUC tous clients": all_auc.mean(), "σ tous": all_auc.std(ddof=1),
                          "AUC clients actifs": act_auc.mean(), "σ actifs": act_auc.std(ddof=1),
                          "_all": all_auc, "_act": act_auc})
leak = pd.DataFrame(leak_rows)
display(leak.drop(columns=["_all", "_act"]).set_index(["modèle", "version"]).round(4))

2 meilleurs modèles : ['CatBoost', 'LightGBM']


AUC tous clients  σ tous  AUC clients actifs  σ actifs
modèle   version                                                           
CatBoost complet               0.6929  0.0038              0.6908    0.0039
         neutralisé            0.6938  0.0041              0.6924    0.0044
LightGBM complet               0.6917  0.0043              0.6896    0.0045
         neutralisé            0.6902  0.0042              0.6891    0.0044

In [ ]:
decision_rows = []
for label in candidates:
    full = leak[(leak["modèle"] == label) & (leak["version"] == "complet")].iloc[0]
    neut = leak[(leak["modèle"] == label) & (leak["version"] == "neutralisé")].iloc[0]
    loss_all = 1000 * (full["_all"] - neut["_all"])
    loss_act = 1000 * (full["_act"] - neut["_act"])
    threshold = 1000 * full["σ tous"]
    decision_rows.append({
        "modèle": label,
        "perte tous clients (millièmes)": f"{loss_all.mean():+.1f} ± {loss_all.std(ddof=1):.1f}",
        "perte clients actifs (millièmes)": f"{loss_act.mean():+.1f} ± {loss_act.std(ddof=1):.1f}",
        "écart-type AUC entre folds (millièmes)": round(threshold, 1),
        "perte < écart-type ?": bool(loss_all.mean() < threshold),
        "écart complet : tous - actifs (millièmes)": round(1000 * (full["AUC tous clients"] - full["AUC clients actifs"]), 1),
    })
leak_decision = pd.DataFrame(decision_rows).set_index("modèle")
leak_decision

,perte tous clients (millièmes),perte clients actifs (millièmes),écart-type AUC entre folds (millièmes),perte < écart-type ?,écart complet : tous - actifs (millièmes)
modèle,,,,,
CatBoost,-1.0 ± 1.1,-1.6 ± 1.3,3.8,True,2.1
LightGBM,+1.5 ± 0.2,+0.6 ± 0.2,4.3,True,2.1


**Lecture** :

- **La neutralisation ne coûte presque rien** : LightGBM perd **1,5 ± 0,2 millième**, CatBoost en gagne même 1,0 ± 1,1 (du bruit). Les deux pertes sont **inférieures à l'écart-type de l'AUC entre folds** (4,3 et 3,8 millièmes).
- **Sur les clients actifs** (99,1 % des clients), l'AUC du modèle complet est plus basse de 2,1 millièmes que sur tous les clients : les 718 clients sans `change_*` sont faciles à classer (76 % de churn). La perte due à la neutralisation n'y est que de 0,6 millième (LightGBM).
- **Conclusion** : le signal « déjà parti » existe (ces clients churnent à 69-76 %), mais il **pèse au plus 1,5 millième d'AUC** : le modèle ne repose pas dessus. Il ne s'agit donc pas d'une fuite qui gonflerait artificiellement la performance.
- **Décision (règle fixée avant les résultats, D67)** : perte inférieure à l'écart-type → **on garde la version complète** et on documente. L'AUC « clients actifs » est conservée comme **métrique complémentaire** en E9, pour vérifier que la performance tient sur les clients qu'une campagne de rétention peut encore toucher.

## 8. Régression logistique : indicateurs en double

Dans la LR, 13 indicateurs `manquant_*` doublonnent une autre colonne : le `missingindicator_*` créé par `add_indicator` (source numérique) ou la modalité « Manquant » du one-hot (source catégorielle). Le coefficient se partage alors entre deux colonnes identiques. On les retire (`dedupe_missing=True`) et on vérifie que l'AUC ne change pas.

In [ ]:
dedup = cross_validate_model(build_pipeline("logreg", dedupe_missing=True), train, y, folds, "LR sans doublons")
diff = 1000 * (dedup.per_fold["roc_auc"].to_numpy() - lr_auc)
n_cols = {label: len(build_pipeline("logreg", **kw).fit(train.iloc[folds[0][0]], y.iloc[folds[0][0]])
                     .named_steps["preprocess"].get_feature_names_out())
          for label, kw in [("avec doublons", {}), ("sans doublons", {"dedupe_missing": True})]}
print(f"AUC avec doublons : {lr_auc.mean():.4f} ; sans doublons : {dedup.per_fold['roc_auc'].mean():.4f}")
print(f"écart fold par fold : {diff.mean():+.2f} ± {diff.std(ddof=1):.2f} millièmes ; colonnes : {n_cols}")

AUC avec doublons : 0.6676 ; sans doublons : 0.6677
écart fold par fold : +0.00 ± 0.04 millièmes ; colonnes : {'avec doublons': 202, 'sans doublons': 189}


**Lecture** : retirer les 13 indicateurs en double (202 → 189 colonnes) ne change pas l'AUC (0,6676 → 0,6677 ; écart fold par fold de +0,00 ± 0,04 millième). La version sans doublons est retenue pour la LR : même performance, et chaque absence d'information n'a plus qu'**un seul** coefficient, lisible sans le partager avec une colonne identique.

## 9. Désignation des finalistes pour E9

Critères : AUC, stabilité entre folds, surapprentissage, calibration (Brier sur l'échantillon équilibré, avant correction du prior), temps d'entraînement et interprétabilité. Le gagnant n'est **pas** choisi ici : il le sera en E9, sur le jeu de test.

In [ ]:
INTERPRETABILITY = {"LR améliorée": "coefficients lisibles", "Arbre de décision": "règles lisibles",
                    "Forêt aléatoire": "SHAP nécessaire", "LightGBM": "SHAP nécessaire", "CatBoost": "SHAP nécessaire"}
criteria = comparison.drop(index=["Dummy", "Règles E4"])[
    ["auc", "auc_std", "écart train-validation", "Brier", "entraînement (s / fold)", "folds gagnés sur 5"]].copy()
criteria["interprétabilité"] = criteria.index.map(INTERPRETABILITY)
criteria.sort_values("auc", ascending=False).round(4)

,auc,auc_std,écart train-validation,Brier,entraînement (s / fold),folds gagnés sur 5,interprétabilité
modèle,,,,,,,
CatBoost,0.6929,0.0038,0.1007,0.2213,48.8706,5,SHAP nécessaire
LightGBM,0.6917,0.0043,0.0772,0.2216,1.0399,5,SHAP nécessaire
Forêt aléatoire,0.6781,0.0055,0.1074,0.2266,9.0876,5,SHAP nécessaire
LR améliorée,0.6676,0.0038,0.0038,0.2286,2.6892,0,coefficients lisibles
Arbre de décision,0.6391,0.0023,0.0076,0.2341,2.5263,0,règles lisibles


**Finalistes pour E9 : LightGBM et la régression logistique améliorée (sans doublons).**

| Critère | LightGBM | CatBoost | LR améliorée |
|---|---|---|---|
| AUC | 0,692 ± 0,004 | 0,693 ± 0,004 | 0,668 ± 0,004 |
| Stabilité entre folds | 5/5 folds devant la LR | 5/5 folds devant la LR | référence |
| Surapprentissage (écart train-validation) | +0,077 | +0,101 | **+0,004** |
| Brier (échantillon équilibré) | 0,222 | 0,221 | 0,229 |
| Entraînement par fold | **1,0 s** | 48,9 s | 2,7 s |
| Interprétabilité | SHAP nécessaire | SHAP nécessaire | **coefficients lisibles** |

- **LightGBM plutôt que CatBoost** : AUC indiscernables (1,2 millième d'écart, sous l'écart-type), mais LightGBM est **49 fois plus rapide** et surapprend moins (+0,077 contre +0,101). Pour le réglage des hyperparamètres (E9), la vitesse compte : on pourra explorer plus de configurations. CatBoost reste disponible dans `build_pipeline`.
- **La LR améliorée comme second finaliste** : 25 millièmes d'AUC en moins, mais un modèle **sans surapprentissage**, **lisible** (chaque coefficient se relie à l'EDA) et simple à maintenir. E9 devra trancher sur ce compromis : ce que coûtent 25 millièmes d'AUC en clients ciblés à tort, contre ce que vaut une explication directe.
- Forêt aléatoire et arbre de décision sont écartés : dominés par le boosting en AUC, sans gain d'interprétabilité par rapport à la LR (forêt) ou avec une AUC trop basse (arbre).